# 08 - EfficientNetB0 XAI-Corrected Experiment

This experiment is driven by the XAI audit from `07_xai_gradcam.ipynb`.

Hypotheses from XAI:

- The model may use texture/color shortcuts rather than decay evidence.
- Some predictions may depend on background or image context.
- Some high-confidence errors suggest deployment risk.

Targeted fix:

- stronger translation/zoom/rotation to reduce background dependence
- brightness/contrast/noise augmentation to reduce color/texture shortcuts
- mild targeted oversampling, no class weights
- fine-tune top EfficientNetB0 layers at low learning rate

This notebook does not overwrite `models/best_model.keras`. It saves a new candidate model under `models/experiments/`.


## 1. Project setup


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(f"Could not find src/ from current directory: {NOTEBOOK_DIR}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


## 2. Imports and configuration


In [ ]:
import json
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay

from src.config import (
    CLASS_NAMES_PATH,
    CM_DIR,
    FIGURES_DIR,
    IMAGE_SIZE,
    MODELS_DIR,
    NUM_CLASSES,
    RANDOM_SEED,
    GROUPED_SPLITS_DIR,
)

# Use grouped source-image splits to avoid offline-augmentation leakage.
SPLITS_DIR = GROUPED_SPLITS_DIR
from src.data.augmentations import build_xai_corrected_augmentation
from src.data.dataloaders import make_dataset_from_dataframe
from src.data.prepare_dataset import targeted_oversample_dataframe
from src.models.efficientnetb0_model import build_efficientnetb0_model, unfreeze_efficientnetb0_top
from src.train.evaluate import evaluate_model
from src.train.train import compile_model, train_model

tf.keras.utils.set_random_seed(RANDOM_SEED)

EXPERIMENT_NAME = "efficientnetb0_xai_corrected"
TARGET_MIN_COUNT = 500
PRETRAINED_WEIGHTS = "imagenet"

HEAD_EPOCHS = 8
HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_EPOCHS = 4
FINE_TUNE_LEARNING_RATE = 1e-5
FINE_TUNE_LAYERS = 40

EXPERIMENT_MODELS_DIR = MODELS_DIR / "experiments"
EXPERIMENT_MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CM_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Experiment:", EXPERIMENT_NAME)
print("Input size:", IMAGE_SIZE)


## 3. Load splits and class names


In [ ]:
train_csv = SPLITS_DIR / "train.csv"
val_csv = SPLITS_DIR / "val.csv"
test_csv = SPLITS_DIR / "test.csv"

for path in [train_csv, val_csv, test_csv, CLASS_NAMES_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}. Run 01_eda.ipynb first.")

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

with open(CLASS_NAMES_PATH, "r", encoding="utf-8-sig") as f:
    class_names = json.load(f)

class_to_index = {name: idx for idx, name in enumerate(class_names)}

for split_df in [train_df, val_df, test_df]:
    if "class_index" not in split_df.columns:
        split_df["class_index"] = split_df["class_name"].map(class_to_index)

print("Original train/val/test:", len(train_df), len(val_df), len(test_df))
print("Classes:", len(class_names))
assert len(class_names) == NUM_CLASSES


## 4. Mild targeted oversampling

Only training rows are oversampled. Validation and test remain untouched.


In [ ]:
original_counts = train_df["class_name"].value_counts().reindex(class_names)
train_oversampled_df = targeted_oversample_dataframe(
    train_df,
    target_min_count=TARGET_MIN_COUNT,
    class_column="class_name",
    random_state=RANDOM_SEED,
)

oversampled_counts = train_oversampled_df["class_name"].value_counts().reindex(class_names)
oversampling_summary = pd.DataFrame({
    "class_name": class_names,
    "original_train_count": original_counts.values,
    "oversampled_train_count": oversampled_counts.values,
})
oversampling_summary["added_rows"] = oversampling_summary["oversampled_train_count"] - oversampling_summary["original_train_count"]
oversampling_summary["oversampled_ratio"] = (oversampling_summary["oversampled_train_count"] / oversampling_summary["original_train_count"]).round(2)

display(oversampling_summary.sort_values("added_rows", ascending=False))
print("Original train rows:", len(train_df))
print("Oversampled train rows:", len(train_oversampled_df))


## 5. Build datasets


In [ ]:
train_ds = make_dataset_from_dataframe(train_oversampled_df, class_to_index, training=True)
val_ds = make_dataset_from_dataframe(val_df, class_to_index, training=False)
test_ds = make_dataset_from_dataframe(test_df, class_to_index, training=False)

for images, labels in train_ds.take(1):
    print("Image batch:", images.shape)
    print("Label batch:", labels.shape)


## 6. Build XAI-corrected EfficientNetB0


In [ ]:
augmentation = build_xai_corrected_augmentation()
model = build_efficientnetb0_model(
    num_classes=NUM_CLASSES,
    augmentation=augmentation,
    dropout_rate=0.35,
    train_base=False,
    weights=PRETRAINED_WEIGHTS,
)
model = compile_model(model, learning_rate=HEAD_LEARNING_RATE)
model.summary()


## 7. Train classifier head


In [ ]:
head_checkpoint_path = EXPERIMENT_MODELS_DIR / f"{EXPERIMENT_NAME}_head_best.keras"

head_history = train_model(
    model,
    train_ds,
    val_ds,
    epochs=HEAD_EPOCHS,
    checkpoint_path=head_checkpoint_path,
)


## 8. Fine-tune top EfficientNetB0 layers

BatchNorm layers remain frozen in `unfreeze_efficientnetb0_top()` to reduce instability.


In [ ]:
fine_tune_checkpoint_path = EXPERIMENT_MODELS_DIR / f"{EXPERIMENT_NAME}_finetuned_best.keras"

model = unfreeze_efficientnetb0_top(model, trainable_layers=FINE_TUNE_LAYERS)
model = compile_model(model, learning_rate=FINE_TUNE_LEARNING_RATE)

fine_tune_history = train_model(
    model,
    train_ds,
    val_ds,
    epochs=FINE_TUNE_EPOCHS,
    checkpoint_path=fine_tune_checkpoint_path,
)


## 9. Training curves


In [ ]:
head_history_df = pd.DataFrame(head_history.history)
head_history_df["stage"] = "head"
fine_tune_history_df = pd.DataFrame(fine_tune_history.history)
fine_tune_history_df["stage"] = "fine_tune"
history_df = pd.concat([head_history_df, fine_tune_history_df], ignore_index=True)
display(history_df)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df.index, history_df["loss"], label="train")
plt.plot(history_df.index, history_df["val_loss"], label="val")
plt.title("Loss")
plt.xlabel("Epoch index")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df.index, history_df["accuracy"], label="train")
plt.plot(history_df.index, history_df["val_accuracy"], label="val")
plt.title("Accuracy")
plt.xlabel("Epoch index")
plt.legend()

plt.tight_layout()
curves_path = FIGURES_DIR / f"{EXPERIMENT_NAME}_training_curves.png"
plt.savefig(curves_path, dpi=150)
plt.show()


## 10. Evaluate on test set


In [ ]:
results = evaluate_model(model, test_ds, class_names)
report = results["classification_report"]
cm = results["confusion_matrix"]

report_df = pd.DataFrame(report).transpose()
display(report_df)

accuracy = float(report["accuracy"])
macro_f1 = float(report["macro avg"]["f1-score"])
weighted_f1 = float(report["weighted avg"]["f1-score"])

print("Test accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))
print("Weighted F1:", round(weighted_f1, 4))


## 11. High-confidence error audit

This is a deployment-risk metric, not just a training metric.


In [ ]:
audit_df = test_df.copy().reset_index(drop=True)
audit_df["y_true"] = results["y_true"]
audit_df["y_pred"] = results["y_pred"]
audit_df["predicted_class"] = [class_names[idx] for idx in audit_df["y_pred"]]
audit_df["correct"] = audit_df["class_name"] == audit_df["predicted_class"]

probabilities = model.predict(test_ds, verbose=1)
audit_df["confidence"] = np.max(probabilities, axis=1)
audit_df["high_confidence_error"] = (~audit_df["correct"]) & (audit_df["confidence"] >= 0.95)

print("Errors:", int((~audit_df["correct"]).sum()))
print("High-confidence errors:", int(audit_df["high_confidence_error"].sum()))
display(audit_df[audit_df["high_confidence_error"]][["class_name", "predicted_class", "confidence", "image_path"]].head(20))


## 12. Confusion matrix


In [ ]:
fig, ax = plt.subplots(figsize=(14, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=90, colorbar=False)
plt.title("EfficientNetB0 XAI-corrected")
plt.tight_layout()
cm_path = CM_DIR / f"{EXPERIMENT_NAME}_confusion_matrix.png"
plt.savefig(cm_path, dpi=150)
plt.show()


## 13. Save artifacts


In [ ]:
model_path = EXPERIMENT_MODELS_DIR / f"{EXPERIMENT_NAME}.keras"
metadata_path = EXPERIMENT_MODELS_DIR / f"{EXPERIMENT_NAME}_metadata.json"
report_path = FIGURES_DIR / f"{EXPERIMENT_NAME}_classification_report.csv"
history_path = FIGURES_DIR / f"{EXPERIMENT_NAME}_history.csv"
audit_path = FIGURES_DIR / f"{EXPERIMENT_NAME}_audit.csv"
oversampling_summary_path = EXPERIMENT_MODELS_DIR / f"{EXPERIMENT_NAME}_oversampling_summary.csv"

model.save(model_path)
report_df.to_csv(report_path)
history_df.to_csv(history_path, index=False)
audit_df.to_csv(audit_path, index=False)
oversampling_summary.to_csv(oversampling_summary_path, index=False)

metadata = {
    "model_name": EXPERIMENT_NAME,
    "base_model_family": "EfficientNetB0",
    "experiment_type": "xai_guided_corrected_training",
    "hypotheses": ["background_bias", "texture_bias", "class_confusion", "unstable_attention"],
    "targeted_fixes": ["stronger_geometric_augmentation", "brightness_contrast_jitter", "gaussian_noise", "deeper_fine_tuning"],
    "num_classes": NUM_CLASSES,
    "image_size": list(IMAGE_SIZE),
    "head_epochs_requested": HEAD_EPOCHS,
    "head_learning_rate": HEAD_LEARNING_RATE,
    "fine_tune_epochs_requested": FINE_TUNE_EPOCHS,
    "fine_tune_learning_rate": FINE_TUNE_LEARNING_RATE,
    "fine_tune_layers": FINE_TUNE_LAYERS,
    "augmentation": "xai_corrected_augmentation",
    "class_weighting": None,
    "oversampling": {
        "enabled": True,
        "strategy": "targeted_min_count_with_replacement",
        "target_min_count": TARGET_MIN_COUNT,
        "original_train_rows": int(len(train_df)),
        "oversampled_train_rows": int(len(train_oversampled_df)),
    },
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "metrics": {
        "test_accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "errors": int((~audit_df["correct"]).sum()),
        "high_confidence_errors": int(audit_df["high_confidence_error"].sum()),
    },
    "artifacts": {
        "model": str(model_path),
        "head_checkpoint": str(head_checkpoint_path),
        "fine_tune_checkpoint": str(fine_tune_checkpoint_path),
        "classification_report": str(report_path),
        "history": str(history_path),
        "audit": str(audit_path),
        "confusion_matrix": str(cm_path),
        "training_curves": str(curves_path),
    },
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved model:", model_path)
print("Saved metadata:", metadata_path)
print("Did not overwrite models/best_model.keras")


## 14. Decision rule

This corrected model is better only if it improves or preserves macro F1 while reducing high-confidence errors and producing more plausible XAI attention in notebook 09.
